In [4]:
# Hot Spot Analysis

import geopandas as gpd
import pandas as pd
import numpy as np
import libpysal
from esda.getisord import G_Local
from tqdm import tqdm

print("=== Improved Hot Spot Analysis on H3 Grid ===")

# Load the H3 grid
hex_gdf = gpd.read_file("accra_solar_h3_grid_ready.gpkg")
print(f"Loaded {len(hex_gdf):,} hexagons")

# Use normalized variable (solar per building) - much better for hotspots
hex_gdf["solar_per_building"] = hex_gdf["total_solar_kwh"] / hex_gdf["building_count"]

variable = "solar_per_building"

# Create spatial weights (Queen contiguity)
w = libpysal.weights.Queen.from_dataframe(hex_gdf, silence_warnings=True)
w.transform = 'r'

# Run Getis-Ord Gi*
print("Computing Getis-Ord Gi* hotspots (normalized)...")
g_local = G_Local(hex_gdf[variable], w, star=True, keep_simulations=True, 
                  transform='r', permutations=999)

# Add results
hex_gdf["gi_zscore"] = g_local.Zs
hex_gdf["gi_pvalue"] = g_local.p_sim

# Multi-level classification
hex_gdf["hotspot_class"] = "Not Significant"

# Hot Spots (positive)
hex_gdf.loc[(hex_gdf["gi_zscore"] > 2.58) & (hex_gdf["gi_pvalue"] < 0.05), "hotspot_class"] = "Hot Spot (99% confidence)"
hex_gdf.loc[(hex_gdf["gi_zscore"] > 1.96) & (hex_gdf["gi_pvalue"] < 0.05), "hotspot_class"] = "Hot Spot (95% confidence)"
hex_gdf.loc[(hex_gdf["gi_zscore"] > 1.65) & (hex_gdf["gi_pvalue"] < 0.05), "hotspot_class"] = "Hot Spot (90% confidence)"

# Cold Spots (negative)
hex_gdf.loc[(hex_gdf["gi_zscore"] < -2.58) & (hex_gdf["gi_pvalue"] < 0.05), "hotspot_class"] = "Cold Spot (99% confidence)"
hex_gdf.loc[(hex_gdf["gi_zscore"] < -1.96) & (hex_gdf["gi_pvalue"] < 0.05), "hotspot_class"] = "Cold Spot (95% confidence)"
hex_gdf.loc[(hex_gdf["gi_zscore"] < -1.65) & (hex_gdf["gi_pvalue"] < 0.05), "hotspot_class"] = "Cold Spot (90% confidence)"

print("\nHotspot Class Distribution:")
print(hex_gdf["hotspot_class"].value_counts())

# Save
hex_gdf.to_file("accra_solar_h3_with_hotspots.gpkg", driver="GPKG")
print("\nSaved improved version as 'accra_solar_h3_with_hotspots.gpkg'")

=== Improved Hot Spot Analysis on H3 Grid ===
Loaded 3,135 hexagons


I:\Temporary\ipykernel_16428\2666961630.py:22: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = libpysal.weights.Queen.from_dataframe(hex_gdf, silence_warnings=True)


Computing Getis-Ord Gi* hotspots (normalized)...
('WARNING: ', 154, ' is an island (no neighbors)')
('WARNING: ', 1173, ' is an island (no neighbors)')
('WARNING: ', 1174, ' is an island (no neighbors)')


C:\Users\Frank\miniconda3\envs\solar_ghana\lib\site-packages\esda\getisord.py:421: UserWarning: Gi* requested, but (a) weights are already row-standardized, (b) no weights are on the diagonal, and (c) no default value supplied to star. Assuming that the self-weight is equivalent to the maximum weight in the row. To use a different default (like, .5), set `star=.5`, or use libpysal.weights.fill_diagonal() to set the diagonal values of your weights matrix and use `star=None` in Gi_Local.
  w, star = _infer_star_and_structure_w(w, star, transform)
C:\Users\Frank\miniconda3\envs\solar_ghana\lib\site-packages\libpysal\weights\util.py:826: UserWarning: The weights matrix is not fully connected: 
 There are 5 disconnected components.
 There are 3 islands with ids: 154, 1173, 1174.
  w = W(neighbors, weights, ids, **kwargs)
C:\Users\Frank\miniconda3\envs\solar_ghana\lib\site-packages\esda\getisord.py:527: RuntimeWarning: invalid value encountered in divide
  z_scores = (statistic - expected_va


Hotspot Class Distribution:
hotspot_class
Not Significant              3091
Hot Spot (90% confidence)      44
Name: count, dtype: int64

Saved improved version as 'accra_solar_h3_with_hotspots.gpkg'
